# Cognexa user study: adaptive vs non-adaptive recommendations

Report Objective 5. This notebook reads the anonymised export from **Admin → User study → Export
anonymised CSV** (`cognexa-study-YYYYMMDD.csv`) and compares the **normalized learning gain**
g = (post − pre) / (max − pre) between the two groups.

Run it with the evaluation dependencies:

```
uv run --project apps/api --group eval jupyter notebook evaluation/study_analysis.ipynb
```

(or open it in VS Code with the `apps/api` virtual environment). If the CSV is not found, a clearly
labelled **synthetic** sample is generated so every cell can be tried; never report synthetic numbers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

CSV = Path("cognexa-study.csv")  # put the exported file next to this notebook (or change the path)
SYNTHETIC = not CSV.exists()
if SYNTHETIC:
    rng = np.random.default_rng(7)
    n = 24
    pre = rng.integers(4, 12, size=2 * n)
    gain_true = np.r_[rng.normal(0.35, 0.18, n), rng.normal(0.25, 0.18, n)]
    post = np.clip(np.round(pre + gain_true * (20 - pre)), 0, 20)
    df = pd.DataFrame(
        {
            "participant": [f"synthetic{i:03d}" for i in range(2 * n)],
            "group": ["adaptive"] * n + ["baseline"] * n,
            "pretest": pre,
            "posttest": post,
            "max_score": 20,
        }
    )
    df["normalized_gain"] = (df.posttest - df.pretest) / (df.max_score - df.pretest)
    print("WARNING: synthetic demo data — do not report these numbers")
else:
    df = pd.read_csv(CSV)
df.head()

## 1. Who completed the study

Participants without both tests have no gain and are excluded from the comparison (reported, not hidden).

In [ ]:
complete = df.dropna(subset=["pretest", "posttest", "normalized_gain"])
summary = (
    df.groupby("group")
    .agg(enrolled=("participant", "count"))
    .join(complete.groupby("group").agg(completed=("participant", "count")))
)
summary["completion_rate"] = summary.completed / summary.enrolled
summary

In [ ]:
complete.groupby("group")[["pretest", "posttest", "normalized_gain"]].describe().T

## 2. Normalized gain: adaptive vs baseline

Welch's t-test (unequal variances) and the Mann-Whitney U test (no normality assumption), plus Cohen's d with the pooled standard deviation.

In [ ]:
a = complete.loc[complete.group == "adaptive", "normalized_gain"].to_numpy()
b = complete.loc[complete.group == "baseline", "normalized_gain"].to_numpy()

welch = stats.ttest_ind(a, b, equal_var=False)
mwu = stats.mannwhitneyu(a, b, alternative="two-sided")
pooled = np.sqrt(
    ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
)
cohen_d = (a.mean() - b.mean()) / pooled
ci = stats.ttest_ind(a, b, equal_var=False).confidence_interval(0.95)

results = pd.Series(
    {
        "n adaptive": len(a),
        "n baseline": len(b),
        "mean gain adaptive": a.mean(),
        "mean gain baseline": b.mean(),
        "difference": a.mean() - b.mean(),
        "95% CI low": ci.low,
        "95% CI high": ci.high,
        "Welch t": welch.statistic,
        "Welch p": welch.pvalue,
        "Mann-Whitney U": mwu.statistic,
        "Mann-Whitney p": mwu.pvalue,
        "Cohen's d": cohen_d,
    }
)
results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120)
complete.boxplot(column="normalized_gain", by="group", ax=axes[0], grid=False)
axes[0].set_title("Normalized gain by group")
axes[0].set_xlabel("")
fig.suptitle("")
for group, color in (("adaptive", "#7c3aed"), ("baseline", "#7a8597")):
    rows = complete[complete.group == group]
    axes[1].scatter(rows.pretest, rows.posttest, label=group, alpha=0.7, color=color)
axes[1].plot([0, 20], [0, 20], color="#999", linewidth=1, linestyle="--")
axes[1].set_xlabel("Pre-test")
axes[1].set_ylabel("Post-test")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

## 3. Engagement (does the group change behaviour?)

In [ ]:
ENGAGEMENT = [
    "quizzes_taken",
    "tutor_questions",
    "recommendations_shown",
    "recommendations_clicked",
    "recommendations_completed",
    "active_days",
]
cols = [c for c in ENGAGEMENT if c in df]
if cols:
    display(df.groupby("group")[cols].median())
    if {"recommendations_clicked", "recommendations_shown"} <= set(df):
        ctr = df.recommendations_clicked / df.recommendations_shown.replace(0, np.nan)
        display(ctr.groupby(df.group).describe())

## 4. Power analysis: plan recruitment honestly

Participants needed **per group** for a two-sided Welch/Student t-test at α = 0.05, by effect size and power. A medium effect is d = 0.5.

In [ ]:
def n_per_group(d, alpha=0.05, power=0.8):
    # smallest n with power >= target, using the noncentral t distribution
    for n in range(4, 2000):
        df_ = 2 * n - 2
        t_crit = stats.t.ppf(1 - alpha / 2, df_)
        nc = d * np.sqrt(n / 2)
        achieved = 1 - stats.nct.cdf(t_crit, df_, nc) + stats.nct.cdf(-t_crit, df_, nc)
        if achieved >= power:
            return n
    return None


table = pd.DataFrame(
    {f"power {p}": [n_per_group(d, power=p) for d in (0.2, 0.5, 0.8)] for p in (0.8, 0.9)},
    index=["small d=0.2", "medium d=0.5", "large d=0.8"],
)
table

In [ ]:
def achieved_power(d, n_a, n_b, alpha=0.05):
    df_ = n_a + n_b - 2
    t_crit = stats.t.ppf(1 - alpha / 2, df_)
    nc = d * np.sqrt(n_a * n_b / (n_a + n_b))
    return 1 - stats.nct.cdf(t_crit, df_, nc) + stats.nct.cdf(-t_crit, df_, nc)


print(
    f"With the current sample (n={len(a)} vs {len(b)}), power to detect d=0.5: "
    f"{achieved_power(0.5, len(a), len(b)):.2f}"
)

## 5. Results paragraph (template for the report)

Fill in from the cells above; keep the wording honest about the sample size and power.

> We enrolled **N** students (adaptive: **n₁**, baseline: **n₂**); **k** completed both tests.
> The adaptive group's mean normalized gain was **g₁** (SD **s₁**) versus **g₂** (SD **s₂**) for
> the baseline group, a difference of **Δ** (95 % CI **[l, u]**). Welch's t-test gave
> t = **t**, p = **p**; the Mann-Whitney U test gave U = **U**, p = **p′**; Cohen's d = **d**.
> With **k** completers the study had a power of **π** to detect a medium effect (d = 0.5), so
> [a non-significant result does not show the absence of an effect / the difference is unlikely
> to be due to chance]. Engagement: the adaptive group followed **x %** of recommendations versus
> **y %** for the baseline.